# Subscription Changes

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce

spark = SparkSession.builder.getOrCreate()

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Databricks Spark session ready")
print(f"[INFO] Batch ID: {batch_id}")

# 1. Read Bronze subscription_changes

In [0]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/subscription_changes"

df_bronze = spark.read.format("delta").load(bronze_path)
df_bronze.show(5)

# 2. Deduplication (change_id + change_date)

In [0]:
w = Window.partitionBy("change_id", "change_date").orderBy(F.col("ingest_time").desc())

df_ranked = df_bronze.withColumn("rn", F.row_number().over(w))

df1 = df_ranked.filter("rn = 1").drop("rn")
df1_quarantine = df_ranked.filter("rn > 1").drop("rn")

# 3. Null validation

In [0]:
required_cols = [
    "change_id", "subscription_id", "old_plan_id",
    "new_plan_id", "change_type", "change_date", "ingest_time"
]

null_condition = None
for c in required_cols:
    cond = F.col(c).isNull()
    null_condition = cond if null_condition is None else (null_condition | cond)

df2 = df1.filter(~null_condition)
df2_quarantine = df1.filter(null_condition)

# 4. Type casting

In [0]:
df3 = df2.select(
    F.col("change_id").cast("bigint"),
    F.col("subscription_id").cast("bigint"),
    F.col("old_plan_id").cast("int"),
    F.col("new_plan_id").cast("int"),
    F.col("change_type").cast("string"),
    F.col("change_date").cast("timestamp"),
    F.col("ingest_time").cast("timestamp")
)

# 5. Validate change_type

In [0]:
allowed_change_types = ["upgrade", "initial", "downgrade"]

df_tmp = df3.withColumn("change_type_clean", F.lower(F.trim(F.col("change_type"))))

df4 = df_tmp.filter(F.col("change_type_clean").isin(allowed_change_types)) \
            .drop("change_type") \
            .withColumnRenamed("change_type_clean", "change_type")

df4_quarantine = df_tmp.filter(~F.col("change_type_clean").isin(allowed_change_types) | F.col("change_type_clean").isNull()) \
                      .withColumn("quarantine_reason", F.lit("invalid change_type")) \
                      .drop("change_type") \
                      .withColumnRenamed("change_type_clean", "change_type")

# 6. Validate plan_id integrity

In [0]:
plan_ref = spark.read.format("delta") \
    .load("/Volumes/datalake_catalog/datalake_schema/silver/plans") \
    .select("plan_id")

df5 = df4.join(plan_ref.withColumnRenamed("plan_id", "old_plan_id"), "old_plan_id", "left_semi") \
         .join(plan_ref.withColumnRenamed("plan_id", "new_plan_id"), "new_plan_id", "left_semi")

df5_quarantine = df4.join(df5.select(df4.columns), df4.columns, "left_anti")

# 7. Combine quarantine tables

In [0]:
df_quarantine_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    [df1_quarantine, df2_quarantine, df4_quarantine, df5_quarantine]
)

print(f"[INFO] Total quarantined rows: {df_quarantine_all.count()}")

# 8. Write to Silver Delta Lake (Upsert)

In [0]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/subscription_changes"

df_upsert = df5

w = Window.partitionBy("change_id").orderBy(F.col("change_date").desc(), F.col("ingest_time").desc())

df_upsert = df_upsert.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

if DeltaTable.isDeltaTable(spark, silver_path):
    target = DeltaTable.forPath(spark, silver_path)

    target.alias("t").merge(
        df_upsert.alias("s"),
        "t.change_id = s.change_id AND t.change_date = s.change_date"
    ).whenMatchedUpdateAll(
        condition="s.ingest_time > t.ingest_time"
    ).whenNotMatchedInsertAll().execute()

else:
    df_upsert.write.format("delta").mode("overwrite").save(silver_path)

print("[SUCCESS] Silver subscription_changes updated")

# 9. Validate Silver Output

In [0]:
spark.read.format("delta").load(silver_path).show(5)